# NavState IMU EKF Example

This notebook demonstrates using the NavState-specific EKF (NavStateImuEKF) with synthetic IMU data generated from a simple scenario. We compare estimated yaw/pitch/roll (YPR), velocity (x,y,z), and position (x,y,z) against ground truth.

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/NavStateImuEKFExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [82]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab  # type: ignore
    %pip install --quiet gtsam-develop
except Exception:
    pass  # Not in Colab

In [83]:
import numpy as np
import plotly.graph_objects as go
import pandas as pd

import gtsam
from gtsam import NavState, Rot3
from gtsam import ConstantTwistScenario, ScenarioRunner

# Utility helpers
def rot3_ypr_rad(R: Rot3):
    return np.degrees(R.ypr())

## Configure Scenario and EKF

In [84]:
# --- Scenario: camera orbiting a fixed point ---
radius = 30.0
angular_velocity = np.pi  # rad/sec
w_b = np.array([0, 0, -angular_velocity])
v_n = np.array([radius * angular_velocity, 0, 0])

scenario = ConstantTwistScenario(w_b, v_n)


In [85]:
# Simulation parameters
dt = 1.0 / 180.0 # 1 degree per step
T = 0.7   # total duration (s)
N = int(T / dt)
params = gtsam.PreintegrationParams.MakeSharedD(9.81)
params.setAccelerometerCovariance(np.diag([1e-3] * 3))
params.setIntegrationCovariance(np.diag([1e-3] * 3))
params.setGyroscopeCovariance(np.diag([1e-4] * 3))
runner = ScenarioRunner(scenario, params, dt, gtsam.imuBias.ConstantBias(np.zeros(3), np.zeros(3)))


## Simulate IMU and run EKF

In [86]:
# Initialize EKF
X0: NavState = scenario.navState(0.0)
P0 = np.eye(9) * 0.1
ekf = gtsam.NavStateImuEKF(X0, P0, params)

In [87]:
times = np.linspace(0.0, T, N + 1)

# Storage
ypr_true, ypr_est = [], []
vel_true, vel_est = [], []
pos_true, pos_est = [], []

# Std-dev storage (from EKF covariance)
rot_std_deg_list = []  # yaw/pitch/roll std in degrees
pos_std_list = []      # x/y/z std in meters
vel_std_list = []      # vx/vy/vz std in m/s

Xk = X0
for k, t in enumerate(times):
    # Ground truth state
    X_true: NavState = scenario.navState(t)
    ypr_true.append(rot3_ypr_rad(X_true.attitude()))
    vel_true.append(np.asarray(X_true.velocity()))
    pos_true.append(np.asarray(X_true.position()))

    if k == 0:
        # Store initial estimate before any prediction
        X_est = ekf.state()
        ypr_est.append(rot3_ypr_rad(X_est.attitude()))
        vel_est.append(np.asarray(X_est.velocity()))
        pos_est.append(np.asarray(X_est.position()))
        # Capture initial covariance std-devs
        P = ekf.covariance()
        std = np.sqrt(np.diag(P))
        rot_std_deg_list.append(np.rad2deg(std[0:3]))
        pos_std_list.append(std[3:6])
        vel_std_list.append(std[6:9])
        continue

    # Simulate noise-free IMU readings
    omega_meas = runner.actualAngularVelocity(t - dt)  # at start of interval
    acc_meas = runner.actualSpecificForce(t - dt)

    # EKF predict
    ekf.predict(omega_meas, acc_meas, dt)
    X_est = ekf.state()
    ypr_est.append(rot3_ypr_rad(X_est.attitude()))
    vel_est.append(np.asarray(X_est.velocity()))
    pos_est.append(np.asarray(X_est.position()))

    # Record covariance std-devs after prediction
    P = ekf.covariance()
    std = np.sqrt(np.diag(P))
    rot_std_deg_list.append(np.rad2deg(std[0:3]))
    pos_std_list.append(std[3:6])
    vel_std_list.append(std[6:9])

    # Every so many states, make a measurement of the position
    if k % 10 == 0:
        predicted_position = X_est.position()
        H = np.zeros((3, 9))
        H[:, :3] = X_est.attitude().matrix()
        measured_position = X_true.position()
        R = np.eye(3) * 1  # measurement noise
        ekf.updateWithVector(predicted_position, H, measured_position, R)
        print(f"Position before update: {predicted_position}")
        print(f"Position after update: {ekf.state().position()}")
        
        # Innovation: y = z - h(x_pred)
        innovation = measured_position - predicted_position

        # Innovation covariance: S = H @ P @ H.T + R
        P = ekf.covariance()
        print(f"P: {(1*P).astype(int)}")
        S = H @ P @ H.T + R

        # Kalman Gain: K = P @ H.T @ np.linalg.inv(S)
        K = P @ H.T @ np.linalg.inv(S)
        print(f"K: {(1*K).astype(int)}")

        # Correction vector: delta_xi = K @ innovation
        delta_xi = K @ innovation
        print(f"Time {t:.2f}s, Innovation: {innovation}\n, Correction: {delta_xi}")
        print(X_est.expmap(delta_xi))

# Convert to arrays
ypr_true = np.unwrap(np.vstack(ypr_true), axis=0)
ypr_est = np.unwrap(np.vstack(ypr_est), axis=0)
vel_true = np.vstack(vel_true)
vel_est = np.vstack(vel_est)
pos_true = np.vstack(pos_true)
pos_est = np.vstack(pos_est)

rot_std_deg = np.vstack(rot_std_deg_list)
pos_std = np.vstack(pos_std_list)
vel_std = np.vstack(vel_std_list)

Position before update: [ 5.21329099 -0.45597705  0.        ]
Position after update: [ 5.21330075e+00 -4.55925577e-01 -1.56500654e-03]
P: [[ 0  0  0  0  0  0  0  0 -1]
 [ 0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  1  0  0]
 [ 0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0]
 [ 0  0  1  0  0  0 24  2  0]
 [ 0  0  0  0  0  0  2  0  0]
 [-1  0  0  0  0  0  0  0 24]]
K: [[ 0  0  0]
 [ 0  0  0]
 [ 0  0  0]
 [ 0  0  0]
 [ 0  0  0]
 [ 0  0  0]
 [ 0  0  1]
 [ 0  0  0]
 [-1  0  0]]
Time 0.06s, Innovation: [ 0.03740078 -0.00709291  0.        ]
, Correction: [ 3.17216793e-03 -4.08833817e-05  0.00000000e+00  6.18904390e-07
  4.80211903e-05 -1.43458329e-03  2.22807623e-05  1.72877870e-03
 -5.18086494e-02]
R: [
	0.984808, 0.173647, -0.000591102;
	-0.173648, 0.984803, -0.00311687;
	4.08833e-05, 0.00317216, 0.999995
]
p:     5.2133  -0.455928 -0.0014345
v:    92.9591   -16.3763 -0.0518058

Position before update: [ 1.02761768e+01 -1.81076778e+00 -6.28379295e-

## Plot Yaw/Pitch/Roll (degrees)

In [88]:
# YPR (deg) with ±2σ bands, RGB colors, dashed GT
import numpy as np
import plotly.graph_objects as go

fig = go.Figure()

# Colors for components
color_rgb = {
    0: ('rgba(255,0,0,0.15)', '#ff0000'),   # yaw -> red
    1: ('rgba(0,128,0,0.15)', '#008000'),   # pitch -> green
    2: ('rgba(0,0,255,0.15)', '#0000ff'),   # roll -> blue
}

names = {0: 'Yaw', 1: 'Pitch', 2: 'Roll'}

for i in range(3):
    fill_rgba, line_color = color_rgb[i]
    mean = ypr_est[:, i]
    std2p = 2.0 * rot_std_deg[:, i]
    upper = mean + std2p
    lower = mean - std2p

    # Shaded band
    fig.add_scatter(x=times, y=upper, name=f"{names[i]} +2σ", line=dict(color=line_color, width=0), showlegend=False)
    fig.add_scatter(x=times, y=lower, name=f"{names[i]} -2σ", fill='tonexty', fillcolor=fill_rgba, line=dict(color=line_color, width=0), showlegend=False)

    # Estimate (solid) and Ground truth (dashed)
    fig.add_scatter(x=times, y=mean, name=f"{names[i]} est", line=dict(color=line_color, width=2))
    fig.add_scatter(x=times, y=ypr_true[:, i], name=f"{names[i]} true", line=dict(color=line_color, dash='dash', width=2))

fig.update_layout(title='YPR (deg) with ±2σ bands', xaxis_title='Time (s)', yaxis_title='Degrees')
fig.show()

## Plot Velocity (m/s)

In [89]:
# Velocity (m/s) with ±2σ bands
import numpy as np
import plotly.graph_objects as go

fig = go.Figure()

color_rgb = {
    0: ('rgba(255,0,0,0.15)', '#ff0000'),   # x -> red
    1: ('rgba(0,128,0,0.15)', '#008000'),   # y -> green
    2: ('rgba(0,0,255,0.15)', '#0000ff'),   # z -> blue
}

for i, comp in enumerate(['x','y','z']):
    fill_rgba, line_color = color_rgb[i]
    mean = vel_est[:, i]
    std2p = 2.0 * vel_std[:, i]
    upper = mean + std2p
    lower = mean - std2p

    fig.add_scatter(x=times, y=upper, name=f"v{comp} +2σ", line=dict(color=line_color, width=0), showlegend=False)
    fig.add_scatter(x=times, y=lower, name=f"v{comp} -2σ", fill='tonexty', fillcolor=fill_rgba, line=dict(color=line_color, width=0), showlegend=False)

    fig.add_scatter(x=times, y=mean, name=f"v{comp} est", line=dict(color=line_color, width=2))
    fig.add_scatter(x=times, y=vel_true[:, i], name=f"v{comp} true", line=dict(color=line_color, dash='dash', width=2))

fig.update_layout(title='Velocity (m/s) with ±2σ bands', xaxis_title='Time (s)', yaxis_title='m/s')
fig.show()

## Plot Position (m)

In [90]:
# Position (m) with ±2σ bands
import numpy as np
import plotly.graph_objects as go

fig = go.Figure()

color_rgb = {
    0: ('rgba(255,0,0,0.15)', '#ff0000'),   # x -> red
    1: ('rgba(0,128,0,0.15)', '#008000'),   # y -> green
    2: ('rgba(0,0,255,0.15)', '#0000ff'),   # z -> blue
}

for i, comp in enumerate(['x','y','z']):
    fill_rgba, line_color = color_rgb[i]
    mean = pos_est[:, i]
    std2p = 2.0 * pos_std[:, i]
    upper = mean + std2p
    lower = mean - std2p

    fig.add_scatter(x=times, y=upper, name=f"p{comp} +2σ", line=dict(color=line_color, width=0), showlegend=False)
    fig.add_scatter(x=times, y=lower, name=f"p{comp} -2σ", fill='tonexty', fillcolor=fill_rgba, line=dict(color=line_color, width=0), showlegend=False)

    fig.add_scatter(x=times, y=mean, name=f"p{comp} est", line=dict(color=line_color, width=2))
    fig.add_scatter(x=times, y=pos_true[:, i], name=f"p{comp} true", line=dict(color=line_color, dash='dash', width=2))

fig.update_layout(title='Position (m) with ±2σ bands', xaxis_title='Time (s)', yaxis_title='m')
fig.show()